# AtlasForge in five minutes — no model, no token, no GPU

This notebook runs the whole workflow on **synthetic demo data**. Every number it produces comes
from fixed rules in `atlasforge demo`, not from N-ATLaS or any other model, so it works on a free
Colab runtime and offline.

By the end you will have seen:

1. a run scored under both tone views,
2. every failure-mode flag,
3. two runs compared example by example, including a **regression** — and
4. a self-contained HTML report rendered inside the notebook.

To do this against the real models, see the last cell.

In [ ]:
# notebook: colab-only
# Colab has no AtlasForge yet. Once v0.1.0 is on PyPI this line is `%pip install -q brainers-atlasforge`.
%pip install -q "git+https://github.com/im-aderm/atlasforge"

import atlasforge

print("atlasforge", atlasforge.__version__)

## 1. The demo data

`build_demo` writes a small dataset (92 items, English plus Yoruba) and two finished runs, `base`
and `tuned`. The answers are computed by fixed rules, and the manifests say so.

The data is shaped to show what a report looks like when the news is mixed: the tuned model is
better overall **and worse at arithmetic**.

In [ ]:
from pathlib import Path

from atlasforge.demo import build_demo

demo = build_demo(Path("atlasforge-demo"))
dataset = demo / "toy_qa.jsonl"
base, tuned = demo / "runs" / "base", demo / "runs" / "tuned"

print(f"dataset:  {dataset}")
print(f"examples: {sum(1 for _ in dataset.open(encoding='utf-8'))}")
print(f"runs:     {base.name}, {tuned.name}")

## 2. Score one run

`write_reports` scores the answers the run already recorded — it calls no model — and writes
`report.md`, `report.json` and `report.html` beside them.

Two things to notice in the output below. Metrics are given under **both tone views**, because
Yoruba tone marks are meaningful and a model that drops them is not free of error. And failed
calls are counted as **wrong**, never hidden: the count is in the footer.

In [ ]:
from atlasforge import write_reports

write_reports(base, dataset)
print((base / "report.md").read_text(encoding="utf-8"))

## 3. Compare two runs

`compare_runs` pairs the two runs example by example on the same dataset. A paired test asks a
much sharper question than two separate scores: did *the same items* move?

The slices are where the interesting part is. Here the tuned model improves overall and
**regresses by 20 points on `has_number`** — a change nobody would have noticed from a single
aggregate score.

In [ ]:
from atlasforge import compare_runs, write_comparison

comparison = compare_runs(dataset, base, tuned, slice_fields=["domain", "has_number"])
write_comparison(comparison, "comparison")
print(Path("comparison/comparison.md").read_text(encoding="utf-8"))

## 4. The HTML report

Every report is also written as one self-contained HTML file: charts as inline SVG, no JavaScript,
nothing fetched from the network. It opens from the filesystem, works air-gapped, and can be
emailed as a single attachment.

In Colab the cell below renders it in place. Run outside a notebook and it just says where the
file is.

In [ ]:
page = Path("comparison/comparison.html")
html = page.read_text(encoding="utf-8")

try:
    from IPython.display import HTML, display
except ImportError:  # running as a plain script
    print(f"{len(html):,} characters of self-contained HTML in {page.name}")
else:
    display(HTML(html))

## 5. Doing this against the real N-ATLaS

Nothing above needed a model. This last cell is the same workflow against a real one, and it is
commented out because it needs things a free Colab runtime does not have: a GPU (or a served
endpoint), an `HF_TOKEN`, and the Awarri licence accepted on each `NCAIR1` model page.

The `openai` backend talks to any OpenAI-compatible server, so the usual shape is to serve N-ATLaS
with vLLM, llama.cpp or Ollama and point `base_url` at it. AtlasForge never routes a request to a
different company's model, and there is no model-as-judge: the model in the report is the model
that produced the answers.

See [the quickstart](https://im-aderm.github.io/atlasforge/get-started/quickstart/) and
[serving N-ATLaS](https://im-aderm.github.io/atlasforge/guides/serve-n-atlas/).

In [ ]:
# notebook: colab-only
# import os
# from atlasforge import evaluate
#
# os.environ["HF_TOKEN"] = "hf_..."  # never paste a token into a notebook you will share
#
# evaluation = evaluate(
#     dataset,
#     out_dir="runs/natlas",
#     backend="openai",
#     base_url="http://127.0.0.1:8000/v1",  # your vLLM / llama.cpp / Ollama endpoint
#     model="NCAIR1/N-ATLaS",
# )
# print(evaluation.metric("exact_match"))